# Pacific War Wikipedia Corpus crawler

Builds the **Corpus**: `Data/corpus.json` = `{article title: raw wikitext}` for every English Wikipedia
Article about the **US–Japan engagement** in WWII (vocabulary: see `CONTEXT.md`).

**How it runs** – one streaming pipeline, backed by a persistent cache (`Data/interim/cache.sqlite`):

```
category crawl ─► fetch pages ─► Wikidata claims ─► class ancestors ─► tag + battle check
      ▲                │                                                     │ In-scope Battle
      └── (seeds)      └──► e5 embeddings (GPU thread)          expansion (P607) + redirects
                                                                             │
                        fetch pages ◄────────────── new titles ◄─────────────┘
```
Every arrow is a batch submitted to a worker pool as soon as its input exists, so crawling,
fetching, Wikidata and embedding overlap. Only the final pass (anchoring, e5 Gray-zone decision,
overrides, export) waits for everything. Every network result is cached; a re-run within
`REFRESH_DAYS` makes no network calls for anything it has already seen.

Config lives in `Processing/config/`: `categories.yaml` (seeds, blocked), `scope.yaml`
(Umbrella conflicts, Relevance queries), `overrides.yaml` (hand include/exclude),
`gold_set.yaml` (recall test), `gray_labels.yaml` (hand labels that calibrate and test Relevance).

In [1]:
from pathlib import Path

ROOT = Path.cwd().parent if Path.cwd().name == "Processing" else Path.cwd()
CONFIG_DIR = ROOT / "Processing" / "config"
INTERIM = ROOT / "Data" / "interim"
CORPUS_PATH = ROOT / "Data" / "corpus.json"
CACHE_PATH = INTERIM / "cache.sqlite"
INTERIM.mkdir(parents=True, exist_ok=True)

REFRESH_DAYS = 30  # cached network results older than this are fetched again
WP_WORKERS = 4  # concurrent Wikipedia API requests
WDQS_WORKERS = 3  # concurrent Wikidata Query Service requests (WDQS allows 5 per IP)
MIN_BATTLE_LINKS = 2  # distinct In-scope Battles an Article must link to (≥1 not an Umbrella conflict)

E5_MODEL = "intfloat/e5-base-v2"
E5_REVISION = "f52bf8ec8c7124536f0efb74aca902b2995e5bcd"  # pinned: scores must not drift between runs
E5_TEXT = "war-sentences-v1"  # bump when evidence_text() changes, so cached vectors are recomputed
E5_TARGET_PRECISION = 0.9  # auto-accept / auto-reject only where calibration labels are this pure

USER_AGENT = "Corpus2GeoRAG_Pacific/0.2 (https://github.com/NivGreenstein/Corpus2GeoRAG_Pacific)"

## Helpers: HTTP, cache, config

In [2]:
import collections
import csv
import json
import re
import sqlite3
import threading
import time
from concurrent.futures import FIRST_COMPLETED, ThreadPoolExecutor, wait

import mwparserfromhell
import numpy as np
import requests
import torch
import yaml

WP_API = "https://en.wikipedia.org/w/api.php"
WDQS = "https://query.wikidata.org/sparql"

_local = threading.local()


def _session():
    if not hasattr(_local, "session"):
        _local.session = requests.Session()
        _local.session.headers["User-Agent"] = USER_AGENT
    return _local.session


def _get(url, params, attempts=6):
    for attempt in range(attempts):
        try:
            r = _session().get(url, params=params, timeout=120)
            if r.status_code in (429, 500, 502, 503, 504):
                raise requests.HTTPError(f"HTTP {r.status_code}")
            r.raise_for_status()
            data = r.json()
            if data.get("error", {}).get("code") == "maxlag":
                time.sleep(int(r.headers.get("Retry-After", 5)))
                continue
            return data
        except (requests.RequestException, ValueError) as e:
            wait_s = 2**attempt
            print(f"  retry in {wait_s}s: {e}")
            time.sleep(wait_s)
    raise RuntimeError(f"giving up on {url}")


def wp_query(**params):
    """Yield the `query` part of each page of a (continued) MediaWiki API query."""
    base = {"action": "query", "format": "json", "formatversion": 2, "maxlag": 5, **params}
    cont = {}
    while True:
        data = _get(WP_API, {**base, **cont})
        yield data.get("query", {})
        if "continue" not in data:
            return
        cont = data["continue"]


def sparql(query):
    data = _get(WDQS, {"query": query, "format": "json"})
    return [{k: v["value"].rsplit("/", 1)[-1] if v["type"] == "uri" else v["value"] for k, v in b.items()}
            for b in data["results"]["bindings"]]


def batched(items, n):
    items = list(items)
    for i in range(0, len(items), n):
        yield items[i:i + n]


def write_jsonl(path, rows):
    with path.open("w") as f:
        for row in rows:
            f.write(json.dumps(row, ensure_ascii=False) + "\n")


class Cache:
    """SQLite store for every network result. Used from the main thread only."""

    def __init__(self, path):
        self.db = sqlite3.connect(path)
        self.db.execute("PRAGMA journal_mode=WAL")  # lets you inspect the cache while a run is writing
        self.db.executescript("""
            CREATE TABLE IF NOT EXISTS kv (ns TEXT, key TEXT, value TEXT, fetched_at REAL, PRIMARY KEY (ns, key));
            CREATE TABLE IF NOT EXISTS wikitext (title TEXT PRIMARY KEY, revid INTEGER, wikitext TEXT);
            CREATE TABLE IF NOT EXISTS vectors (title TEXT, variant TEXT, revid INTEGER, vec BLOB,
                                                PRIMARY KEY (title, variant));
            DROP TABLE IF EXISTS embeddings;
        """)
        self.fresh_after = time.time() - REFRESH_DAYS * 86400

    def get(self, ns, key):
        row = self.db.execute("SELECT value, fetched_at FROM kv WHERE ns=? AND key=?", (ns, key)).fetchone()
        return json.loads(row[0]) if row and row[1] >= self.fresh_after else None

    def put(self, ns, key, value):
        self.db.execute("INSERT OR REPLACE INTO kv VALUES (?,?,?,?)", (ns, key, json.dumps(value), time.time()))

    def put_wikitext(self, title, revid, text):
        self.db.execute("INSERT OR REPLACE INTO wikitext VALUES (?,?,?)", (title, revid, text))

    def get_wikitext(self, title):
        row = self.db.execute("SELECT wikitext FROM wikitext WHERE title=?", (title,)).fetchone()
        return row[0] if row else ""

    def get_embedding(self, title, revid):
        row = self.db.execute("SELECT vec FROM vectors WHERE title=? AND variant=? AND revid=?",
                              (title, E5_VARIANT, revid)).fetchone()
        return np.frombuffer(row[0], dtype=np.float32) if row else None

    def put_embedding(self, title, revid, vec):
        self.db.execute("INSERT OR REPLACE INTO vectors VALUES (?,?,?,?)",
                        (title, E5_VARIANT, revid, np.asarray(vec, dtype=np.float32).tobytes()))

    def commit(self):
        self.db.commit()


categories_cfg = yaml.safe_load((CONFIG_DIR / "categories.yaml").read_text())
scope_cfg = yaml.safe_load((CONFIG_DIR / "scope.yaml").read_text())
overrides = yaml.safe_load((CONFIG_DIR / "overrides.yaml").read_text())
gold = yaml.safe_load((CONFIG_DIR / "gold_set.yaml").read_text())
gold_tag = {t: tag for tag, titles in gold.items() for t in titles}
UMBRELLA = set(scope_cfg["umbrella_conflicts"])
QUERIES = scope_cfg["relevance_queries"]
gray_labels = yaml.safe_load((CONFIG_DIR / "gray_labels.yaml").read_text())
E5_VARIANT = f"{E5_MODEL}|{E5_TEXT}"
FORCE_INCLUDE, FORCE_EXCLUDE = set(overrides.get("include") or []), set(overrides.get("exclude") or [])
print(f"{len(categories_cfg['seeds'])} seeds, {len(gold_tag)} gold articles, {len(UMBRELLA)} umbrella conflicts, "
      f"{len(FORCE_INCLUDE)}+{len(FORCE_EXCLUDE)} overrides")

25 seeds, 55 gold articles, 10 umbrella conflicts, 1103+729 overrides


## Network tasks
Each function runs in a worker thread, does network (and per-page parsing) only, and returns
plain data; the main thread owns all state and the cache.

In [3]:
import sys

sys.path.insert(0, str(ROOT / "Processing"))
from evidence import REF_RE, evidence_text, sentences
from scope_rules import CLAIM_PROPS, ANCHOR_PROPS, article_links, norm_title

YEAR_RE = re.compile(r"\b(1[89]\d\d|20\d\d)\b")
SKIP_INFOBOX_PARAMS = re.compile(r"^(partof|image|caption|image_size|alt|map\w*|module|footnotes|notes)$")


def extract_features(title, wikitext):
    lead = wikitext.split("\n==", 1)[0]
    parsed = mwparserfromhell.parse(lead)
    infobox_name, infobox_text, infobox_years = None, "", []
    for tpl in parsed.filter_templates(recursive=False):
        name = str(tpl.name).strip()
        if name.lower().startswith("infobox"):
            infobox_name = name[len("infobox"):].strip(" _").lower()
            infobox_text = REF_RE.sub("", " ".join(
                str(p.value) for p in tpl.params if not SKIP_INFOBOX_PARAMS.match(str(p.name).strip())))
            if tpl.has("date"):
                infobox_years = sorted({int(y) for y in YEAR_RE.findall(str(tpl.get("date").value))})
            break
    lead_text = re.sub(r"\s+", " ", REF_RE.sub("", parsed.strip_code())).strip()[:3000]
    links = article_links(wikitext)
    return {"infobox": infobox_name, "infobox_text": infobox_text, "infobox_years": infobox_years,
            "links": links, "lead_text": lead_text}


def task_category(cat):
    members = {"pages": [], "subcats": []}
    for q in wp_query(list="categorymembers", cmtitle=cat, cmtype="page|subcat", cmnamespace="0|14", cmlimit="max"):
        for m in q.get("categorymembers", []):
            members["subcats" if m["ns"] == 14 else "pages"].append(m["title"])
    return members


def task_fetch(titles):
    """-> (aliases {requested: canonical or None}, [page record incl. wikitext])"""
    pages, texts, aliases = {}, {}, {}
    for q in wp_query(titles="|".join(titles), redirects=1, prop="revisions|categories|pageprops",
                      rvprop="ids|content", rvslots="main", cllimit="max", clshow="!hidden",
                      ppprop="wikibase_item|disambiguation"):
        for r in q.get("normalized", []) + q.get("redirects", []):
            aliases[r["from"]] = r["to"]
        for p in q.get("pages", []):
            if p.get("missing") or p.get("invalid") or p.get("ns") != 0 or "disambiguation" in p.get("pageprops", {}):
                continue
            rec = pages.setdefault(p["title"], {"title": p["title"], "pageid": p["pageid"], "revid": None,
                                                "qid": p.get("pageprops", {}).get("wikibase_item"), "categories": []})
            rec["categories"] += [c["title"] for c in p.get("categories", [])]
            if p.get("revisions"):
                rec["revid"] = p["revisions"][0]["revid"]
                texts[p["title"]] = p["revisions"][0]["slots"]["main"]["content"]
    resolved = {}
    for t in titles:
        canon = t
        while canon in aliases:
            canon = aliases[canon]
        resolved[t] = canon if canon in pages else None
    for title, rec in pages.items():
        rec.update(extract_features(title, texts.get(title, "")))
        rec["wikitext"] = texts.get(title, "")
    return resolved, list(pages.values())


def task_claims(qids):
    out = {q: {p: [] for p in CLAIM_PROPS} for q in qids}
    values = " ".join(f"wd:{q}" for q in qids)
    unions = " UNION ".join(f'{{ ?item wdt:{p} ?val BIND("{p}" AS ?prop) }}' for p in CLAIM_PROPS)
    for r in sparql(f"SELECT ?item ?prop ?val WHERE {{ VALUES ?item {{ {values} }} {unions} }}"):
        out[r["item"]][r["prop"]].append(r["val"])
    return out


def task_ancestors(classes):
    """Full superclass closure per class (matching to Tag roots happens locally, so roots can change freely)."""
    out = {c: [c] for c in classes}
    values = " ".join(f"wd:{c}" for c in classes)
    for r in sparql(f"SELECT ?c ?s WHERE {{ VALUES ?c {{ {values} }} ?c wdt:P279* ?s }}"):
        out[r["c"]].append(r["s"])
    return out


def task_expand(battle_qids):
    """Articles whose Wikidata conflict / participant-in / significant event is one of the battles -> {battle qid: [titles]}."""
    out = {q: [] for q in battle_qids}
    values = " ".join(f"wd:{q}" for q in battle_qids)
    path = "|".join(f"wdt:{p}" for p in ANCHOR_PROPS)
    for r in sparql(f"""SELECT DISTINCT ?b ?title WHERE {{ VALUES ?b {{ {values} }} ?item {path} ?b .
            ?article schema:about ?item; schema:isPartOf <https://en.wikipedia.org/>; schema:name ?title }}"""):
        out[r["b"]].append(r["title"])
    return out


def task_redirects(titles):
    out = {t: [] for t in titles}
    for q in wp_query(titles="|".join(titles), prop="redirects", rdlimit="max", rdnamespace=0):
        for p in q.get("pages", []):
            out.setdefault(p["title"], []).extend(r["title"] for r in p.get("redirects", []))
    return out


_model = None


def e5_model():
    global _model
    if _model is None:
        from sentence_transformers import SentenceTransformer
        device = "mps" if torch.backends.mps.is_available() else "cuda" if torch.cuda.is_available() else "cpu"
        _model = SentenceTransformer(E5_MODEL, revision=E5_REVISION, device=device)
    return _model


def task_embed(items):
    """items: [(title, revid, wikitext)] -> [(title, revid, vector)]. Runs on the single GPU thread."""
    vecs = e5_model().encode([evidence_text(t, w) for t, _, w in items], batch_size=64,
                             normalize_embeddings=True, show_progress_bar=False)
    return [(t, r, v) for (t, r, _), v in zip(items, vecs)]

## Tagging and scope rules

**Tag** (primary): infobox type → Intelligence categories → Wikidata `P31` class → category keywords.
Title patterns refine Battle/Campaign/Operation and Ship/Ship class. `Media` is never in scope.

**In-scope Battle**: a Battle/Campaign/Operation whose participants include both the US and Japan
(infobox, Wikidata `P710`, or the "Battles of World War II involving …" categories), dated 1941–1945.

**Anchored** (everything else, and Battles that fail the participants check, e.g. cancelled operations):
Wikidata `P607` names a non-umbrella In-scope Battle, *or* the Article sits in a Category named after
one, *or* it is tied to ≥ `MIN_BATTLE_LINKS` distinct In-scope Battles of which ≥ 1 is not an Umbrella
conflict. Articles with weaker ties form the **Gray zone**, decided by e5 Relevance.

In [4]:
# The rules live in scope_rules.py (tested by tests/test_scope_rules.py); RULES.md explains them.
from scope_rules import (BATTLE_TAGS, PACIFIC, allied_only, anchor_scope, battle_grade, battle_scope, battle_sides,
                         participants_known, tag_page, too_broad, without_namesake)


## Streaming pipeline
The main thread runs an event loop: whenever a task finishes, its result is cached, merged into state,
and may queue more work (new categories, titles, Wikidata ids, classes, battles, texts to embed).
Queues are flushed into batches: full batches go out immediately; partial batches go out whenever
their worker pool would otherwise sit idle.

In [5]:
class Pipeline:
    BATCH = {"category": 1, "fetch": 50, "claims": 200, "ancestors": 50, "expand": 100, "redirects": 50, "embed": 128}
    POOL = {"category": "wp", "fetch": "wp", "redirects": "wp", "claims": "wdqs", "ancestors": "wdqs",
            "expand": "wdqs", "embed": "gpu"}
    TASK = {"category": task_category, "fetch": task_fetch, "claims": task_claims, "ancestors": task_ancestors,
            "expand": task_expand, "redirects": task_redirects, "embed": task_embed}
    # Cache namespaces whose payload changed shape; bump the suffix when CLAIM_PROPS / ANCHOR_PROPS change.
    CLAIMS_NS, EXPAND_NS = "claims-v2", "expand-v2"

    def __init__(self, cache):
        self.cache = cache
        self.pools = {"wp": ThreadPoolExecutor(WP_WORKERS), "wdqs": ThreadPoolExecutor(WDQS_WORKERS),
                      "gpu": ThreadPoolExecutor(1)}
        self.workers = {"wp": WP_WORKERS, "wdqs": WDQS_WORKERS, "gpu": 1}
        self.pending = {}  # future -> (kind, batch)
        self.queues = {k: [] for k in self.BATCH}
        self.queued = {k: set() for k in self.BATCH}
        self.stats = collections.Counter()
        # crawl state
        self.blocked_exact = set(categories_cfg.get("blocked") or [])
        self.blocked_re = re.compile("|".join(categories_cfg.get("blocked_patterns") or ["$^"]), re.I)
        self.members, self.remaining, self.tree = {}, {}, {}
        # article state
        self.candidates = {}  # requested title -> {"found_via", "seen_in"}
        self.canonical = {}  # requested title -> canonical title (None = missing)
        self.pages, self.claims, self.ancestors, self.embedded = {}, {}, {}, {}
        self.waiting_qid, self.waiting_cls = collections.defaultdict(set), collections.defaultdict(set)
        self.battles, self.battle_aliases = {}, {}

    # ---------- queueing ----------
    def enqueue(self, kind, key):
        if key not in self.queued[kind]:
            self.queued[kind].add(key)
            self.queues[kind].append(key)

    def flush(self):
        busy = collections.Counter(self.POOL[k] for k, _ in self.pending.values())
        for kind, q in self.queues.items():
            pool, n = self.POOL[kind], self.BATCH[kind]
            while q and (len(q) >= n or busy[pool] < self.workers[pool]):
                batch, self.queues[kind] = q[:n], q[n:]
                q = self.queues[kind]
                arg = (batch[0] if kind == "category" else
                       [(t, self.pages[t]["revid"], self.cache.get_wikitext(t)) for t in batch] if kind == "embed" else
                       batch)
                self.pending[self.pools[pool].submit(self.TASK[kind], arg)] = (kind, batch)
                busy[pool] += 1

    # ---------- crawl ----------
    def is_blocked(self, cat):
        return cat in self.blocked_exact or bool(self.blocked_re.search(cat.removeprefix("Category:")))

    def visit(self, cat, depth, remaining, seed, parent):
        if self.remaining.get(cat, -1) >= remaining:
            return
        first = cat not in self.remaining
        self.remaining[cat] = remaining
        if first:
            self.tree[cat] = {"category": cat, "seed": seed, "depth": depth, "parent": parent, "n_articles": 0,
                              "status": "blocked" if self.is_blocked(cat) else "crawled"}
        if self.tree[cat]["status"] == "blocked":
            return
        if cat in self.members:
            self.expand_category(cat)
        elif (hit := self.cache.get("category", cat)) is not None:
            self.on_category(hit, cat, from_cache=True)
        else:
            self.enqueue("category", cat)

    def on_category(self, members, cat, from_cache=False):
        if not from_cache:
            self.cache.put("category", cat, members)
        self.members[cat] = members
        self.tree[cat]["n_articles"] = len(members["pages"])
        self.stats["categories"] += 1
        self.expand_category(cat)

    def expand_category(self, cat):
        row, remaining = self.tree[cat], self.remaining[cat]
        for sub in self.members[cat]["subcats"]:
            if remaining > 0:
                self.visit(sub, row["depth"] + 1, remaining - 1, row["seed"], cat)
            elif sub not in self.tree:
                self.tree[sub] = {"category": sub, "seed": row["seed"], "depth": row["depth"] + 1, "parent": cat,
                                  "status": "beyond depth", "n_articles": None}
        for title in self.members[cat]["pages"]:
            self.add_candidate(title, "category", cat)

    # ---------- pages ----------
    def add_candidate(self, title, via, seen_in=None):
        cand = self.candidates.setdefault(title, {"found_via": via, "seen_in": []})
        if seen_in:
            cand["seen_in"].append(seen_in)
        if title in self.canonical or title in self.queued["fetch"]:
            return
        canon = self.cache.get("alias", title)
        page = canon and self.cache.get("page", canon)
        if canon is not None and (canon == "" or page):
            self.canonical[title] = canon or None
            if page:
                self.on_page(page)
        else:
            self.enqueue("fetch", title)

    def on_fetch(self, result, batch):
        resolved, pages = result
        for p in pages:
            text = p.pop("wikitext")
            self.cache.put_wikitext(p["title"], p["revid"], text)
            self.cache.put("page", p["title"], p)
        for req, canon in resolved.items():
            self.canonical[req] = canon
            self.cache.put("alias", req, canon or "")
        for p in pages:
            self.on_page(p)

    def on_page(self, p):
        if p["title"] in self.pages:
            return
        self.pages[p["title"]] = p
        self.stats["pages"] += 1
        if p["revid"] is not None:
            vec = self.cache.get_embedding(p["title"], p["revid"])
            if vec is not None:
                self.embedded[p["title"]] = vec
            else:
                self.enqueue("embed", p["title"])
        qid = p["qid"]
        if qid and qid not in self.claims and (hit := self.cache.get(self.CLAIMS_NS, qid)) is not None:
            self.claims[qid] = hit
        self.evaluate(p["title"])

    # ---------- Wikidata ----------
    def on_claims(self, result, batch):
        for qid, cl in result.items():
            self.cache.put(self.CLAIMS_NS, qid, cl)
            self.claims[qid] = cl
            for title in self.waiting_qid.pop(qid, set()):
                self.evaluate(title)

    def on_ancestors(self, result, batch):
        for cls, anc in result.items():
            self.cache.put("ancestors", cls, anc)
            self.ancestors[cls] = anc
            for title in self.waiting_cls.pop(cls, set()):
                self.evaluate(title)

    def evaluate(self, title):
        """Tag a page once its claims and class ancestors are known; register In-scope Battles."""
        p = self.pages[title]
        if p["qid"] and p["qid"] not in self.claims:
            self.waiting_qid[p["qid"]].add(title)
            self.enqueue("claims", p["qid"])
            return
        missing = [c for c in self.claims.get(p["qid"], {}).get("P31", []) if c not in self.ancestors] if p["qid"] else []
        for c in missing:
            if (hit := self.cache.get("ancestors", c)) is not None:
                self.ancestors[c] = hit
            else:
                self.waiting_cls[c].add(title)
                self.enqueue("ancestors", c)
        if any(c not in self.ancestors for c in missing):
            return
        tag, _, _ = tag_page(p, self.claims, self.ancestors)
        self.stats["tagged"] += 1
        if tag in BATTLE_TAGS and battle_scope(p, self.claims)[0] == "in" and title not in self.battles:
            self.on_battle(title, p["qid"])

    def on_battle(self, title, qid):
        self.battles[title] = qid
        self.battle_aliases[title] = title
        if (hit := self.cache.get("redirects", title)) is not None:
            self.on_redirects({title: hit}, [title], from_cache=True)
        else:
            self.enqueue("redirects", title)
        if qid:
            if (hit := self.cache.get(self.EXPAND_NS, qid)) is not None:
                self.on_expand({qid: hit}, [qid], from_cache=True)
            else:
                self.enqueue("expand", qid)

    def on_expand(self, result, batch, from_cache=False):
        for qid, titles in result.items():
            if not from_cache:
                self.cache.put(self.EXPAND_NS, qid, titles)
            for t in titles:
                self.add_candidate(t, "wikidata")

    def on_redirects(self, result, batch, from_cache=False):
        for title, aliases in result.items():
            if not from_cache:
                self.cache.put("redirects", title, aliases)
            canon = title if title in self.battles else self.canonical.get(title, title)
            for a in aliases:
                self.battle_aliases[a] = canon

    # ---------- embeddings ----------
    def on_embed(self, result, batch):
        for title, revid, vec in result:
            self.cache.put_embedding(title, revid, vec)
            self.embedded[title] = vec

    # ---------- loop ----------
    def run(self, seeds, default_depth):
        for s in seeds:
            self.visit(s["name"], 0, s.get("depth", default_depth), s["name"], "")
        started, last_report = time.time(), 0
        while True:
            self.flush()
            if not self.pending:
                break
            done, _ = wait(list(self.pending), timeout=5, return_when=FIRST_COMPLETED)
            for fut in done:
                kind, batch = self.pending.pop(fut)
                result = fut.result()
                self.stats[f"{kind} batches"] += 1
                if kind == "category":
                    self.on_category(result, batch[0])
                else:
                    getattr(self, f"on_{kind}")(result, batch)
            if time.time() - last_report > 15:
                self.cache.commit()
                last_report = time.time()
                q = {k: len(v) for k, v in self.queues.items() if v}
                print(f"[{time.time() - started:5.0f}s] categories {self.stats['categories']:,} · pages {len(self.pages):,} · "
                      f"battles {len(self.battles):,} · embedded {len(self.embedded):,} · in flight {len(self.pending)} · queued {q}")
        self.cache.commit()
        for pool in self.pools.values():
            pool.shutdown()
        print(f"done in {time.time() - started:.0f}s: {len(self.tree):,} categories, {len(self.pages):,} pages, "
              f"{len(self.battles):,} in-scope battles, {len(self.embedded):,} embeddings")


cache = Cache(CACHE_PATH)
pipe = Pipeline(cache)
pipe.run(categories_cfg["seeds"], categories_cfg.get("default_depth", 3))

done in 0s: 1,050 categories, 14,665 pages, 320 in-scope battles, 14,664 embeddings


## Classify – rules first, Gray zone left open

In [6]:
claims, ancestors = pipe.claims, pipe.ancestors
# Discovery kept every battle with any US+Japan signal; classification reads the combatant fields side by side.
battle_grades = {t: battle_grade(pipe.pages[t], claims, battle_sides(cache.get_wikitext(t))) for t in pipe.battles}
battles = {t: q for t, q in pipe.battles.items() if battle_grades[t][0] != "out"}
battle_aliases = {a: t for a, t in pipe.battle_aliases.items() if t in battles}
battle_by_qid = {q: t for t, q in battles.items() if q}
# Peripheral battles (US or Japan only in support) and disputed ones, like Umbrella conflicts, never anchor alone.
WEAK_ANCHORS = UMBRELLA | {t for t, (g, _) in battle_grades.items() if g in ("peripheral", "disagree")}
print("In-scope Battles by grade:", dict(collections.Counter(g for g, _ in battle_grades.values())))


def anchor(p, tag):
    return anchor_scope(p, tag, claims, battles, battle_by_qid, battle_aliases, WEAK_ANCHORS, MIN_BATTLE_LINKS)


def classify(p):
    ties = None
    tag, source, all_tags = tag_page(p, claims, ancestors)
    cand = pipe.candidates.get(p["title"], {})
    row = {k: p[k] for k in ("title", "pageid", "revid", "qid", "infobox")}
    row.update(primary_tag=tag, tag_source=source, tags=all_tags, categories=p["categories"],
               found_via=cand.get("found_via", "category"), seen_in=cand.get("seen_in", []))
    if tag == "Media":
        verdict, reason = "out", "media / culture"
    elif tag in BATTLE_TAGS:
        b, reason = battle_scope(p, claims)
        if b == "in":
            grade, reason = battle_grades.get(p["title"]) or battle_grade(p, claims, battle_sides(cache.get_wikitext(p["title"])))
            verdict = {"in": "in", "peripheral": "in", "disagree": "gray", "out": "out"}[grade]
            ties = {"grade": "sources disagree"} if grade == "disagree" else None
        elif b == "dated":
            verdict = "out"
        elif participants_known(p, claims):  # listed, but not the US and Japan
            verdict, reason = "out", "battle without the US and Japan (participants listed)"
        else:  # e.g. cancelled operations: fall back to anchoring
            a, why, ties = anchor(p, tag)
            verdict, reason = {"anchored": "in", "gray": "gray", "none": "out"}[a], f"{reason}; {why}"
    elif broad := too_broad(p, claims, ancestors):
        verdict, reason = "out", f"{broad}: too broad to anchor (use an Override to include)"
    else:
        a, reason, ties = anchor(p, tag)
        verdict = {"anchored": "in" if tag else "gray", "gray": "gray", "none": "out"}[a]
        if verdict != "out":
            wikitext = cache.get_wikitext(p["title"]) or ""
            if allied_only(p, wikitext):
                verdict, reason = "out", "Allied force without the US: " + reason
            elif tag in {"Ship", "Ship class"} and (own := without_namesake(wikitext)) != wikitext:
                a2, _, _ = anchor({**p, "links": article_links(own)}, tag)
                if a2 != "anchored" and not PACIFIC.search(" ".join(sentences(own))):
                    verdict, reason = "out", "tied only through its namesake, no Pacific service: " + reason
    row.update(rule_verdict=verdict, reason=reason, anchor_grade=ties and ties["grade"])
    return row


rows = {t: classify(p) for t, p in pipe.pages.items()}
print(collections.Counter(r["rule_verdict"] for r in rows.values()))

In-scope Battles by grade: {'disagree': 4, 'in': 298, 'peripheral': 13, 'out': 5}


Counter({'out': 7812, 'in': 4088, 'gray': 2765})


## e5 Relevance – decides the Gray zone
Relevance = cosine(war sentences, `in_scope` query) − cosine(war sentences, `out_of_scope` query),
with both queries taken from `scope.yaml`. It is recorded for every Article.

Two thresholds are set from the **calibration** labels in `gray_labels.yaml` (plus any Gray-zone
Overrides): Gray-zone Articles at or above `accept_at` join the Corpus, those below `reject_below`
are left out, and the band in between is left out but listed first in `gray_zone.csv` for review.
The **holdout** labels are only used to report how well the thresholds work.

In [7]:
titles = [t for t in rows if t in pipe.embedded]
E = np.stack([pipe.embedded[t] for t in titles])
q_in, q_out = e5_model().encode([f"query: {QUERIES['in_scope']}", f"query: {QUERIES['out_of_scope']}"],
                                normalize_embeddings=True)
relevance = dict(zip(titles, (E @ q_in - E @ q_out).tolist()))
for t, r in rows.items():
    r["relevance"] = round(relevance[t], 4) if t in relevance else None


def labelled(section):
    labels = {t: bool(y) for t, y in (gray_labels.get(section) or {}).items()}
    if section == "calibration":  # Gray-zone Overrides are labels too
        labels.update({t: t in FORCE_INCLUDE for t in FORCE_INCLUDE | FORCE_EXCLUDE
                       if rows.get(t, {}).get("rule_verdict") == "gray" and t not in (gray_labels.get("holdout") or {})})
    return [(relevance[t], y) for t, y in labels.items() if t in relevance]


def thresholds(points, target=E5_TARGET_PRECISION, min_count=3):
    """accept_at: lowest score whose ≥-side is ≥ target positive; reject_below: highest score whose <-side is ≥ target negative."""
    s = np.array([p for p, _ in points]); y = np.array([l for _, l in points])
    cuts = np.unique(s)
    accept = next((c for c in cuts if (s >= c).sum() >= min_count and y[s >= c].mean() >= target), np.inf)
    reject = next((c for c in cuts[::-1] if (s < c).sum() >= min_count and 1 - y[s < c].mean() >= target), -np.inf)
    if reject > accept:  # one cut satisfies both sides: no review band
        accept = reject = (accept + reject) / 2
    return float(accept), float(reject)


ACCEPT_AT, REJECT_BELOW = thresholds(labelled("calibration"))
print(f"calibration: {len(labelled('calibration'))} labels → accept_at {ACCEPT_AT:+.4f}, reject_below {REJECT_BELOW:+.4f}")
(INTERIM / "thresholds.json").write_text(json.dumps({"accept_at": ACCEPT_AT, "reject_below": REJECT_BELOW}))
hold = labelled("holdout")
auto = [(s, y) for s, y in hold if s >= ACCEPT_AT or s < REJECT_BELOW]
if hold:
    correct = sum((s >= ACCEPT_AT) == y for s, y in auto)
    print(f"holdout: {len(hold)} labels · {len(auto)} decided automatically, {correct} correct "
          f"({correct / max(len(auto), 1):.0%}) · {len(hold) - len(auto)} in the review band")

/Users/nivgreenstein/Desktop/projects/geoai/Corpus2GeoRAG_Pacific/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 16627.82it/s]

calibration: 1134 labels → accept_at +0.0661, reject_below +0.0375
holdout: 68 labels · 54 decided automatically, 50 correct (93%) · 14 in the review band


## kNN – which past decisions does each Article look like?
Neighbours are frozen in `knn_neighbours.yaml`: original review Overrides, calibration labels, the Gold set
and look-alikes, without recovery Overrides. Rebuild only on purpose with `knn_neighbours.py --rebuild`. Holdout, evaluation-set and audit titles are never neighbours, so they can
still measure. kNN never decides alone: it is the second key in the Decide cell (`second_key` in `scope.yaml`).

In [8]:
import knn

eval_cfg = yaml.safe_load((CONFIG_DIR / "eval_set.yaml").read_text()) if (CONFIG_DIR / "eval_set.yaml").exists() else {}
audit_cfg = yaml.safe_load((CONFIG_DIR / "corpus_audit.yaml").read_text()) if (CONFIG_DIR / "corpus_audit.yaml").exists() else {}
frozen = yaml.safe_load((CONFIG_DIR / "knn_neighbours.yaml").read_text())
never = (set(gray_labels.get("holdout") or {}) | {t for s in eval_cfg.values() if isinstance(s, dict) for t in s}
         | {t for s in audit_cfg.values() if isinstance(s, dict) for t in s})
neighbour_labels = {t: side == "in" for side, titles in frozen.items() for t in titles if t not in never}
NEIGHBOURS = knn.fit(pipe.embedded, neighbour_labels)
knn_scores = knn.score(NEIGHBOURS, {t: pipe.embedded[t] for t in rows if t in pipe.embedded})
for t, r in rows.items():
    s, nearest = knn_scores.get(t, (None, []))
    r["knn_score"] = None if s is None else round(s, 3)
    r["knn_neighbours"] = [f"{n} ({'in' if y else 'out'})" for n, y in nearest]
print(f"kNN: {len(NEIGHBOURS.titles):,} neighbours ({int(NEIGHBOURS.y.sum()):,} in scope), K = {NEIGHBOURS.k}")

kNN: 1,308 neighbours (672 in scope), K = 11


## Decide – rules, then e5 for the Gray zone, then overrides → `tagged.jsonl`, `excluded.jsonl`, `untagged.jsonl`, `gray_zone.csv`

In [9]:
TAGGED_PATH, EXCLUDED_PATH = INTERIM / "tagged.jsonl", INTERIM / "excluded.jsonl"
UNTAGGED_PATH, GRAY_PATH = INTERIM / "untagged.jsonl", INTERIM / "gray_zone.csv"
RECOVERY_PATH = INTERIM / "recovery.csv"
SECOND_KEY = scope_cfg.get("second_key") or {}


def second_key(r):
    """e5 and kNN may overturn a rule verdict only when they agree (scope.yaml: second_key)."""
    rel, k = r["relevance"], r.get("knn_score")
    if rel is None or k is None:
        return
    if (SECOND_KEY.get("veto") and r["in_scope"] and r["anchor_grade"] == "links" and r["primary_tag"] not in BATTLE_TAGS
            and rel < REJECT_BELOW and k < SECOND_KEY.get("veto_knn", 0.3)):
        r["in_scope"], r["decided_by"] = False, "e5+knn"
        r["reason"] += "; e5 and kNN both say unrelated → out"
    elif (not r["in_scope"] and r["anchor_grade"] == "none" and r["primary_tag"] not in (None, "Media")
          and rel >= ACCEPT_AT and k >= SECOND_KEY.get("recovery_knn", 0.8)):
        if SECOND_KEY.get("recovery") == "include":
            r["in_scope"], r["decided_by"] = True, "e5+knn"
            r["reason"] += "; e5 and kNN both say related → in"
        else:
            r["decided_by"] = "recovery"  # left out until a person decides



for r in rows.values():
    if r["title"] in FORCE_INCLUDE or r["title"] in FORCE_EXCLUDE:
        r["in_scope"], r["decided_by"] = r["title"] in FORCE_INCLUDE, "override"
    elif r["rule_verdict"] == "gray":
        rel = r["relevance"] if r["relevance"] is not None else -np.inf
        r["decided_by"] = "e5" if rel >= ACCEPT_AT or rel < REJECT_BELOW else "review"
        r["in_scope"] = rel >= ACCEPT_AT
        if r["decided_by"] == "review" and r["anchor_grade"] == "umbrella only":
            # Too close to call and tied only through an Umbrella conflict: left out without review.
            r["decided_by"], r["reason"] = "rules", r["reason"] + "; e5 too close to call → out"
    else:
        r["in_scope"], r["decided_by"] = r["rule_verdict"] == "in", "rules"
        second_key(r)

previous = {json.loads(l)["title"] for l in TAGGED_PATH.open()} if TAGGED_PATH.exists() else set()
tagged = [r for r in rows.values() if r["in_scope"]]
excluded = [r for r in rows.values() if not r["in_scope"] and r["primary_tag"]]
untagged = [r for r in rows.values() if not r["in_scope"] and not r["primary_tag"]]
write_jsonl(TAGGED_PATH, tagged)
write_jsonl(EXCLUDED_PATH, excluded)
write_jsonl(UNTAGGED_PATH, untagged)
gray = [r for r in rows.values() if r["rule_verdict"] == "gray"]
gray.sort(key=lambda r: (r["decided_by"] != "review", -(r["relevance"] if r["relevance"] is not None else -9)))
with GRAY_PATH.open("w", newline="") as f:
    w = csv.writer(f)
    w.writerow(["title", "primary_tag", "relevance", "decision", "reason", "lead"])
    for r in gray:
        decision = "review" if r["decided_by"] == "review" else "accept" if r["in_scope"] else "reject"
        w.writerow([r["title"], r["primary_tag"], r["relevance"], decision, r["reason"],
                    pipe.pages[r["title"]]["lead_text"][:200]])
recovery = sorted((r for r in rows.values() if r["decided_by"] == "recovery"), key=lambda r: -r["knn_score"])
with RECOVERY_PATH.open("w", newline="") as f:
    w = csv.writer(f)
    w.writerow(["title", "primary_tag", "relevance", "knn_score", "nearest_decided", "lead"])
    for r in recovery:
        w.writerow([r["title"], r["primary_tag"], r["relevance"], r["knn_score"], "; ".join(r["knn_neighbours"]),
                    pipe.pages[r["title"]]["lead_text"][:200]])
CHANGES_PATH = INTERIM / "changes.csv"
changed = sorted((r for r in rows.values() if r["in_scope"] != (r["title"] in previous)),
                 key=lambda r: (not r["in_scope"], r["primary_tag"] or "", r["title"]))
vanished = sorted(previous - set(rows))  # in the previous Corpus, not discovered this run
with CHANGES_PATH.open("w", newline="") as f:
    w = csv.writer(f)
    w.writerow(["title", "change", "primary_tag", "decided_by", "reason"])
    for r in changed:
        w.writerow([r["title"], "added" if r["in_scope"] else "removed", r["primary_tag"], r["decided_by"], r["reason"]])
    w.writerows([t, "removed", "", "", "not discovered"] for t in vanished)
added = [r for r in changed if r["in_scope"]]
print(f"vs previous run: +{len(added):,} added, −{len(changed) - len(added) + len(vanished):,} removed → {CHANGES_PATH.name}")
for label, key in (("tag", lambda r: r["primary_tag"]), ("how", lambda r: r["reason"].split(" →")[0].split(":")[0][:40])):
    print(f"  added by {label}:", dict(collections.Counter(map(key, added)).most_common(8)))
by = collections.Counter(r["decided_by"] for r in gray)
print(f"gray zone {len(gray):,}: accepted {sum(r['in_scope'] for r in gray if r['decided_by'] == 'e5'):,}, "
      f"rejected {sum(not r['in_scope'] for r in gray if r['decided_by'] == 'e5'):,}, review band {by['review']:,} (left out)")
print(f"second key: {sum(r['decided_by'] == 'e5+knn' and not r['in_scope'] for r in rows.values()):,} vetoed, "
      f"{sum(r['decided_by'] == 'e5+knn' and r['in_scope'] for r in rows.values()):,} recovered, "
      f"{len(recovery):,} recovery candidates for review → {RECOVERY_PATH.name}")
print(f"in corpus: {len(tagged):,} (rules {sum(r['decided_by'] == 'rules' for r in tagged):,}, "
      f"e5 {sum(r['decided_by'] == 'e5' for r in tagged):,}, override {sum(r['decided_by'] == 'override' for r in tagged):,})"
      f" · excluded {len(excluded):,} · untagged {len(untagged):,}")

vs previous run: +19 added, −21 removed → changes.csv
  added by tag: {'Person': 10, 'Military unit': 4, 'Event': 3, 'Location': 1, 'Ship': 1}
  added by how: {'tied to 2 in-scope battles': 10, 'tied to 3 in-scope battles': 2, 'tied to 7 in-scope battles': 2, 'tied to 23 in-scope battles': 1, 'tied to 5 in-scope battles': 1, 'tied to 8 in-scope battles': 1, 'tied to 4 in-scope battles': 1, 'tied to 10 in-scope battles': 1}
gray zone 2,765: accepted 659, rejected 961, review band 7 (left out)
second key: 157 vetoed, 0 recovered, 9 recovery candidates for review → recovery.csv
in corpus: 5,693 (rules 3,931, e5 659, override 1,103) · excluded 8,824 · untagged 148


## Export – `Data/corpus.json` = `{title: wikitext}`, streamed from the cache

In [10]:
tmp = CORPUS_PATH.with_suffix(".partial")
written = 0
with tmp.open("w") as out:
    out.write("{")
    for r in sorted(tagged, key=lambda r: r["title"]):
        (text,) = cache.db.execute("SELECT wikitext FROM wikitext WHERE title=?", (r["title"],)).fetchone()
        out.write(("," if written else "") + "\n" + json.dumps(r["title"], ensure_ascii=False) + ": "
                  + json.dumps(text, ensure_ascii=False))
        written += 1
    out.write("\n}\n")
tmp.rename(CORPUS_PATH)
print(f"wrote {written:,} articles to {CORPUS_PATH} ({CORPUS_PATH.stat().st_size / 1e6:,.1f} MB)")

wrote 5,693 articles to /Users/nivgreenstein/Desktop/projects/geoai/Corpus2GeoRAG_Pacific/Data/corpus.json (104.3 MB)


## Reports
### Gold-set recall
Every Gold-set Article must be in the Corpus. A different primary tag is a warning, not a failure.

In [11]:
missing = 0
for title, expected in gold_tag.items():
    r = rows.get(title)
    if not r or not r["in_scope"]:
        missing += 1
        print(f"✗ {title}: {'not discovered' if not r else r['reason']} (relevance {r and r['relevance']})")
    elif r["primary_tag"] != expected:
        print(f"~ {title}: tag {r['primary_tag']} (expected {expected})")
print(f"\nrecall: {len(gold_tag) - missing}/{len(gold_tag)}")

~ Doolittle Raid: tag Battle (expected Operation)
~ Surrender of Japan: tag Diplomacy/Document (expected Event)
~ Imperial General Headquarters: tag Military unit (expected Organization)

recall: 55/55


### Category review table → `category_review.csv`
Pass rate = share of a category's Articles that made it into the Corpus. Categories with many
Articles and a very low pass rate are suggested for `blocked` in `categories.yaml`.

In [12]:
REVIEW_PATH = INTERIM / "category_review.csv"
in_corpus = {r["title"] for r in tagged}
members = collections.defaultdict(list)
for req, cand in pipe.candidates.items():
    canon = pipe.canonical.get(req)
    for c in cand["seen_in"]:
        if canon:
            members[c].append(canon)
review = []
for row in pipe.tree.values():
    m = members.get(row["category"], [])
    passed = [t for t in m if t in in_corpus]
    rate = len(passed) / len(m) if m else None
    review.append({**row, "n_fetched": len(m), "n_in_corpus": len(passed),
                   "pass_rate": None if rate is None else round(rate, 3),
                   "suggest_block": bool(m and len(m) >= 20 and rate < 0.05),
                   "sample_in": "; ".join(passed[:3]), "sample_out": "; ".join([t for t in m if t not in in_corpus][:3])})
with REVIEW_PATH.open("w", newline="") as f:
    w = csv.DictWriter(f, fieldnames=list(review[0]))
    w.writeheader()
    w.writerows(review)
print(f"wrote {REVIEW_PATH}")
for r in sorted((r for r in review if r["suggest_block"]), key=lambda r: -r["n_fetched"])[:25]:
    print(f"  suggest block: {r['category']} ({r['n_fetched']} articles, {r['n_in_corpus']} kept)")

wrote /Users/nivgreenstein/Desktop/projects/geoai/Corpus2GeoRAG_Pacific/Data/interim/category_review.csv
  suggest block: Category:Military units and formations in Burma in World War II (204 articles, 6 kept)
  suggest block: Category:Chinese military personnel of World War II (141 articles, 1 kept)
  suggest block: Category:Chinese people of World War II (92 articles, 1 kept)
  suggest block: Category:Politics of the Second Sino-Japanese War (90 articles, 0 kept)
  suggest block: Category:BPUPK (62 articles, 1 kept)
  suggest block: Category:Chinese Civil War (54 articles, 0 kept)
  suggest block: Category:Indonesian collaborators with the Empire of Japan (50 articles, 2 kept)
  suggest block: Category:War crimes in the Second Sino-Japanese War (40 articles, 0 kept)
  suggest block: Category:Members of the Central Advisory Council (33 articles, 0 kept)
  suggest block: Category:Burma Railway (31 articles, 1 kept)
  suggest block: Category:British Malaya in World War II (30 articles, 1

### Tag counts

In [13]:
by_tag = collections.Counter(r["primary_tag"] or "(untagged)" for r in tagged)
for tag, n in by_tag.most_common():
    print(f"{n:6,} {tag}")
print("found via:", dict(collections.Counter(r["found_via"] for r in tagged)))
print("decided by:", dict(collections.Counter(r["decided_by"] for r in tagged)))

 2,721 Ship
 1,431 Person
   359 Military unit
   326 Location
   255 Battle
   142 Ship class
    92 Weapon/Equipment
    89 Operation
    63 Aircraft
    63 Diplomacy/Document
    46 Event
    32 (untagged)
    28 Campaign
    24 Organization
    22 Intelligence
found via: {'category': 5039, 'wikidata': 654}
decided by: {'rules': 3931, 'e5': 659, 'override': 1103}
